# Algoritmos de búsqueda en grafos

En este notebook trabajaremos sobre **un mismo problema** para comparar:

- Depth-First Search (DFS)
- Breadth-First Search (BFS)
- Uniform Cost Search (UCS)
- Greedy Best-First Search (GBF)
- A*

La idea central es observar que la estructura general de búsqueda cambia poco.  
Lo que cambia principalmente es **cómo se organiza la frontier**.

## 1. Problema de búsqueda

- Estado inicial: `A`
- Estado meta: `F`
- Los números en las aristas representan el costo de moverse entre estados.

![Grafo DFS](https://drive.google.com/uc?export=view&id=1Bu3wvTzHffMQVHmGfvekSdmtaatt12qJ)



## 2. Representación del problema

Como el grafo tiene costos, cada vecino se representa mediante una tupla:

```python
(estado_vecino, costo_de_la_arista)
```

El orden de los vecinos también importa para DFS y BFS.

In [8]:
graph = {
    "A": [("B", 2), ("C", 1)],
    "B": [("D", 2), ("E", 1)],
    "C": [("E", 3)],
    "D": [("H", 1)],
    "E": [("H", 2)],
    "H": []
}

start = "A"
goal = "H"

## 3. Estructuras comunes

Cada nodo de búsqueda guarda:

- `state`: estado actual.
- `parent`: nodo desde el cual se llegó.
- `cost`: costo acumulado \(g(n)\).

In [9]:
class Node:
    def __init__(self, state, parent=None, cost=0):
        self.state = state
        self.parent = parent
        self.cost = cost

    def __repr__(self):
        return f"{self.state}(g={self.cost})"

In [11]:
def reconstruct_path(node):
    path = []

    while node is not None:
        path.append(node.state)
        node = node.parent

    return list(reversed(path))

## 4. DFS

DFS utiliza una **pila LIFO**.

Los costos aparecen en el grafo, pero DFS no los utiliza para decidir qué nodo expandir.

In [2]:
class StackFrontier:
    def __init__(self):
        self.frontier = []

    def add(self, node):
        self.frontier.append(node)

    def contains_state(self, state):
        return any(node.state == state for node in self.frontier)

    def empty(self):
        return len(self.frontier) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        return self.frontier.pop()

    def states(self):
        return [node.state for node in self.frontier]

In [12]:
def depth_first_search(graph, start, goal, verbose=True):
    frontier = StackFrontier()
    frontier.add(Node(start))

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node = frontier.remove()
        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo: {node.state}")
            print(f"Frontier antes de agregar vecinos: {frontier.states()}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            if (
                child_state not in explored
                and not frontier.contains_state(child_state)
            ):
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )
                frontier.add(child)

        if verbose:
            print(f"Frontier después de expandir: {frontier.states()}")
            print("-" * 45)

    return None

In [13]:
dfs_result = depth_first_search(graph, start, goal)

print("Orden de expansión:", " → ".join(dfs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(dfs_result["path"]))
print("Costo del camino:", dfs_result["cost"])

Expandiendo: A
Frontier antes de agregar vecinos: []
Frontier después de expandir: ['B', 'C']
---------------------------------------------
Expandiendo: C
Frontier antes de agregar vecinos: ['B']
Frontier después de expandir: ['B', 'E']
---------------------------------------------
Expandiendo: E
Frontier antes de agregar vecinos: ['B']
Frontier después de expandir: ['B', 'H']
---------------------------------------------
Expandiendo: H
Frontier antes de agregar vecinos: ['B']
Orden de expansión: A → C → E → H
Camino encontrado: A → C → E → H
Costo del camino: 6


## 5. BFS — actividad

BFS utiliza una **cola FIFO**.

### Tareas

1. Complete el método `remove` de `QueueFrontier`.
2. Complete `breadth_first_search`.
3. Verifique el orden de expansión y el camino encontrado.
4. Compare el resultado con su solución manual.

In [ ]:
class QueueFrontier(StackFrontier):
    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def breadth_first_search(graph, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Ejecute esta celda cuando complete BFS.
bfs_result = breadth_first_search(graph, start, goal)

print("Orden de expansión:", " → ".join(bfs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(bfs_result["path"]))
print("Costo del camino:", bfs_result["cost"])

In [ ]:
# Pruebas básicas para BFS
assert bfs_result["path"] == ["A", "B", "D", "H"]
assert bfs_result["expansion_order"] == ["A", "B", "C", "D", "E", "H"]

print("✓ BFS pasó las pruebas.")

## 6. UCS

UCS utiliza una **cola de prioridad** y siempre expande el nodo con menor costo acumulado:

$$
g(n)
$$

Si aparece un camino más barato hacia un estado que ya estaba en la frontier, se conserva el de menor costo.

### Frontier con prioridad

En **Uniform Cost Search (UCS)**, **Greedy Best-First Search (GBF)** y **A\*** la *frontier* ya no es una pila ni una cola, sino una **cola de prioridad**.

La prioridad determina cuál será el siguiente nodo en expandirse:

- **UCS:** prioridad = \(g(n)\)
- **GBF:** prioridad = \(h(n)\)
- **A\*:** prioridad = \(g(n)+h(n)\)

Esta implementación utiliza el módulo `heapq` de Python, que mantiene automáticamente el elemento con **menor prioridad** en la primera posición.

#### Componentes de la clase

- **`heap`**: almacena la cola de prioridad.
- **`counter`**: genera un número consecutivo para cada inserción. Se utiliza para desempatar cuando dos nodos tienen la misma prioridad, respetando el orden en que fueron agregados.
- **`add(node, priority)`**: inserta un nodo en la frontier con su prioridad correspondiente.
- **`remove()`**: extrae el nodo con la menor prioridad.
- **`empty()`**: indica si la frontier está vacía.

Cada elemento del heap se almacena como una tupla:

```python
(priority, insertion_order, node)
```

Por ejemplo,

```python
(5, 3, Node("E"))
```

significa:

- prioridad = **5**
- fue el **cuarto nodo** insertado (`3` porque el contador empieza en 0)
- corresponde al nodo **E**.

De esta forma, si dos nodos tienen la misma prioridad, se expandirá primero el que fue insertado antes.

### ¿Qué hace `heapq.heappush`?

La siguiente instrucción agrega un nuevo nodo a la **cola de prioridad** (`heap`):

```python
heapq.heappush(
    self.heap,
    (priority, next(self.counter), node)
)
```

Observa que **no se almacena únicamente el nodo**, sino una **tupla** con tres elementos:

```python
(priority, insertion_order, node)
```

donde:

- **`priority`**: valor utilizado para ordenar la frontier.
  - UCS: `g(n)`
  - GBF: `h(n)`
  - A*: `g(n) + h(n)`

- **`next(self.counter)`**: número consecutivo que indica el orden de inserción.
  Se utiliza para desempatar cuando dos nodos tienen la misma prioridad.

- **`node`**: objeto que contiene el estado, el padre y el costo acumulado.

---

### Ejemplo

Supongamos que insertamos los siguientes nodos:

```python
(5, 0, Node("B"))
(3, 1, Node("C"))
(5, 2, Node("D"))
```

La prioridad es el **primer elemento** de la tupla, por lo que el primer nodo en salir será:

```text
Node("C")
```

porque tiene prioridad **3**.

Posteriormente saldrán:

```text
Node("B")
Node("D")
```

Ambos tienen prioridad **5**, pero `B` fue insertado antes (`0 < 2`).

---

### ¿Por qué usar el contador?

Si almacenáramos únicamente

```python
(priority, node)
```

y dos nodos tuvieran la misma prioridad, Python intentaría comparar directamente los objetos `Node`, lo que produciría un error.

El contador evita este problema y garantiza un criterio de desempate consistente.

In [14]:
import heapq
from itertools import count

In [15]:
class PriorityFrontier:
    def __init__(self):
        self.heap = []
        self.counter = count()

    def add(self, node, priority):
        # El contador permite desempatar respetando el orden de inserción.
        heapq.heappush(
            self.heap,
            (priority, next(self.counter), node)
        )

    def empty(self):
        return len(self.heap) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        priority, _, node = heapq.heappop(self.heap)
        return node, priority

In [16]:
def uniform_cost_search(graph, start, goal, verbose=True):
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=0)

    # Mejor costo conocido para cada estado.
    best_cost = {start: 0}

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        # Ignorar entradas antiguas de la cola de prioridad.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost)

    return None

In [17]:
ucs_result = uniform_cost_search(graph, start, goal)

print("Orden de expansión:", " → ".join(ucs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(ucs_result["path"]))
print("Costo óptimo:", ucs_result["cost"])

Expandiendo A: g=0
Expandiendo C: g=1
Expandiendo B: g=2
Expandiendo E: g=3
Expandiendo D: g=4
Expandiendo H: g=5
Orden de expansión: A → C → B → E → D → H
Camino encontrado: A → B → E → H
Costo óptimo: 5


## 7. Heurística para GBF y A*

La heurística \(h(n)\) estima el costo restante desde cada nodo hasta la meta.

![Heurística](heuristica_busqueda.png)

In [18]:
heuristic = {
    "A": 4,
    "B": 3,
    "C": 2,
    "D": 2,
    "E": 1,
    "H": 0
}

## 8. Greedy Best-First Search — actividad

GBF utiliza únicamente:

$$
h(n)
$$

### Tareas

1. Reutilice `PriorityFrontier`.
2. Use la heurística como prioridad.
3. No use el costo acumulado para decidir qué nodo expandir.
4. Retorne el camino, orden de expansión y costo real del camino.

In [ ]:
def greedy_best_first_search(graph, heuristic, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Ejecute esta celda cuando complete GBF.
gbf_result = greedy_best_first_search(graph, heuristic, start, goal)

print("Orden de expansión:", " → ".join(gbf_result["expansion_order"]))
print("Camino encontrado:", " → ".join(gbf_result["path"]))
print("Costo del camino:", gbf_result["cost"])

In [ ]:
# Pruebas básicas para GBF
assert gbf_result["path"] == ["A", "C", "E", "H"]
assert gbf_result["expansion_order"] == ["A", "C", "E", "H"]
assert gbf_result["cost"] == 6

print("✓ GBF pasó las pruebas.")

## 9. A* — actividad

A* combina costo acumulado y heurística:

$$
f(n)=g(n)+h(n)
$$

### Tareas

1. Adapte la implementación de UCS.
2. Mantenga `best_cost` para permitir mejoras de ruta.
3. Use `new_cost + heuristic[child_state]` como prioridad.
4. Verifique que encuentre un camino óptimo.

In [ ]:
def a_star_search(graph, heuristic, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Ejecute esta celda cuando complete A*.
astar_result = a_star_search(graph, heuristic, start, goal)

print("Orden de expansión:", " → ".join(astar_result["expansion_order"]))
print("Camino encontrado:", " → ".join(astar_result["path"]))
print("Costo óptimo:", astar_result["cost"])

In [ ]:
# Pruebas básicas para A*
assert astar_result["path"] in (
    ["A", "B", "D", "H"],
    ["A", "B", "E", "H"]
)
assert astar_result["cost"] == 5

print("✓ A* pasó las pruebas.")

## 10. Comparación final

Complete esta tabla después de ejecutar todos los algoritmos.

| Algoritmo | Tipo de frontier | Prioridad | Camino | Costo | Orden de expansión |
|---|---|---|---|---:|---|
| DFS | Pila | LIFO |  |  |  |
| BFS | Cola | FIFO |  |  |  |
| UCS | Cola de prioridad | \(g(n)\) |  |  |  |
| GBF | Cola de prioridad | \(h(n)\) |  |  |  |
| A* | Cola de prioridad | \(g(n)+h(n)\) |  |  |  |

### Preguntas de cierre

1. ¿Qué algoritmos garantizan el camino de menor número de aristas?
2. ¿Qué algoritmos garantizan el camino de menor costo?
3. ¿Por qué GBF puede encontrar un camino subóptimo?
4. ¿Qué ocurre con A* si \(h(n)=0\) para todos los nodos?
5. ¿Qué efecto tiene el orden de los vecinos en DFS y BFS?

![Grafo DFS](https://drive.google.com/uc?export=view&id=1C-Gocif6ltAwtFuRoROrR7D5rfs7jjcc)

